In [3]:
nonescape_path = "../../backend/nonescape"

In [5]:
from safetensors.torch import load_file
def load_weights_keys(model_name):
    weights = load_file(f"{nonescape_path}/{model_name}.safetensors")
    with open(f"{model_name}_weights_keys.txt", "w", encoding="utf-8") as f:
        for key in weights.keys():
            f.write(f"{key}\n")
    print(f"Wrote weight keys to {model_name}_weights_keys.txt")


In [6]:
load_weights_keys('nonescape-mini-v0')
load_weights_keys('nonescape-v0')

Wrote weight keys to nonescape-mini-v0_weights_keys.txt
Wrote weight keys to nonescape-v0_weights_keys.txt


In [7]:
from pathlib import Path
import sys
from safetensors.torch import load_file

# Reuse the official model definitions shipped in this repo.
nonescape_python_path = Path("../../backend/nonescape/python").resolve()
if str(nonescape_python_path) not in sys.path:
    sys.path.append(str(nonescape_python_path))

from nonescape import NonescapeClassifier, NonescapeClassifierMini

def infer_variant_from_keys(state_dict):
    keys = list(state_dict.keys())
    if any(k.startswith("vit_backbone.") for k in keys):
        return "full"
    if any(k.startswith("backbone.") for k in keys):
        return "mini"
    return "unknown"

def reconstruct_from_safetensors(model_path):
    state_dict = load_file(str(model_path))
    variant = infer_variant_from_keys(state_dict)

    if variant == "full":
        model = NonescapeClassifier()
    elif variant == "mini":
        model = NonescapeClassifierMini()
    else:
        raise ValueError(f"Could not infer architecture from keys in: {model_path}")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    model.eval()

    print(f"Loaded: {model_path.name}")
    print(f"  inferred variant: {variant}")
    print(f"  missing keys: {len(missing)}")
    print(f"  unexpected keys: {len(unexpected)}")
    if missing:
        print("  sample missing:", missing[:5])
    if unexpected:
        print("  sample unexpected:", unexpected[:5])

    return model

mini_model = reconstruct_from_safetensors(Path(f"{nonescape_path}/nonescape-mini-v0.safetensors"))
full_model = reconstruct_from_safetensors(Path(f"{nonescape_path}/nonescape-v0.safetensors"))

c:\Users\Seth_\Documents\Coding\SlopMop\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: nonescape-mini-v0.safetensors
  inferred variant: mini
  missing keys: 0
  unexpected keys: 0


Loading weights: 100%|██████████| 439/439 [00:00<00:00, 3041.76it/s]


Loaded: nonescape-v0.safetensors
  inferred variant: full
  missing keys: 0
  unexpected keys: 0


In [ ]:
import torch
from torch import nn

def configure_finetune(model, lr=1e-4, weight_decay=1e-4, train_backbone=False):
    # Start from a stable default: train only the classification head.
    for p in model.parameters():
        p.requires_grad = False

    if hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True

    # Optionally unfreeze the backbone for full fine-tuning once head-only converges.
    if train_backbone:
        for p in model.parameters():
            p.requires_grad = True

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    return optimizer, criterion

# Example setup for mini model
optimizer, criterion = configure_finetune(mini_model, lr=2e-4, train_backbone=False)
print(f"Trainable parameters: {sum(p.numel() for p in mini_model.parameters() if p.requires_grad):,}")

Trainable parameters: 2,050


: 